In [1]:
import pandas as pd

# 1. Load the datasets
movies = pd.read_csv(r"C:\Users\DIPTANGSHU\Downloads\movies.csv")
ratings = pd.read_csv(r"C:\Users\DIPTANGSHU\Downloads\ratings.csv")

# 2. Merge them together on the 'movieId' column
movie_data = pd.merge(ratings, movies, on='movieId')

# 3. Display the first 5 rows to see what our data looks like
print("Data Preview:")
print(movie_data.head())

Data Preview:
   userId  movieId  rating  timestamp  \
0       1       17     4.0  944249077   
1       1       25     1.0  944250228   
2       1       29     2.0  943230976   
3       1       30     5.0  944249077   
4       1       32     5.0  943228858   

                                               title  \
0                       Sense and Sensibility (1995)   
1                           Leaving Las Vegas (1995)   
2  City of Lost Children, The (Cité des enfants p...   
3  Shanghai Triad (Yao a yao yao dao waipo qiao) ...   
4          Twelve Monkeys (a.k.a. 12 Monkeys) (1995)   

                                   genres  
0                           Drama|Romance  
1                           Drama|Romance  
2  Adventure|Drama|Fantasy|Mystery|Sci-Fi  
3                             Crime|Drama  
4                 Mystery|Sci-Fi|Thriller  


In [4]:
# 1. Count how many ratings each movie has
movie_rating_counts = movie_data['title'].value_counts()

# 2. Filter movies more aggressively - increase threshold to reduce memory usage
popular_movies = movie_rating_counts[movie_rating_counts >= 1000].index  # Increased from 500 to 1000
filtered_movie_data = movie_data[movie_data['title'].isin(popular_movies)]

# 3. Also filter users who have rated fewer movies to further reduce matrix size
user_rating_counts = filtered_movie_data['userId'].value_counts()
active_users = user_rating_counts[user_rating_counts >= 50].index  # Increased from 20 to 50
filtered_movie_data = filtered_movie_data[filtered_movie_data['userId'].isin(active_users)]

# 4. Create sparse matrix directly without pivot_table to avoid memory issues
from scipy.sparse import coo_matrix
import numpy as np

# Create mappings for users and movies to indices
unique_users = filtered_movie_data['userId'].unique()
unique_movies = filtered_movie_data['title'].unique()

user_to_idx = {user: idx for idx, user in enumerate(unique_users)}
movie_to_idx = {movie: idx for idx, movie in enumerate(unique_movies)}

# Map the data to indices
user_indices = filtered_movie_data['userId'].map(user_to_idx)
movie_indices = filtered_movie_data['title'].map(movie_to_idx)
ratings = filtered_movie_data['rating'].values

# Create sparse matrix directly - much more memory efficient
movie_matrix_sparse = coo_matrix((ratings, (user_indices, movie_indices)), 
                                shape=(len(unique_users), len(unique_movies)))

# Convert to CSR format for better performance in operations
movie_matrix_sparse = movie_matrix_sparse.tocsr()

# 5. Display matrix info
print(f"Matrix shape: {movie_matrix_sparse.shape}")
print(f"Number of non-zero entries: {movie_matrix_sparse.nnz}")
print(f"Sparsity: {(1 - movie_matrix_sparse.nnz / (movie_matrix_sparse.shape[0] * movie_matrix_sparse.shape[1])) * 100:.2f}%")
print("Memory usage significantly reduced by using sparse matrix!")

# 6. If you need to see some actual values, convert a small portion to dense
print("\nSample of the matrix (first 5x5):")
sample_dense = movie_matrix_sparse[:5, :5].toarray()
print(sample_dense)

Matrix shape: (125268, 4398)
Number of non-zero entries: 26178939
Sparsity: 95.25%
Memory usage significantly reduced by using sparse matrix!

Sample of the matrix (first 5x5):
[[4. 1. 2. 5. 5.]
 [0. 0. 0. 0. 0.]
 [5. 0. 0. 0. 0.]
 [0. 0. 0. 0. 4.]
 [0. 0. 0. 0. 3.]]


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Clean the genres string (replace the '|' separator with a space)
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)

# 2. Convert text genres into numerical vectors
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

# Memory-efficient function to get content-based recommendations
def get_content_recommendations(movie_title, top_n=5):
    if movie_title not in movies['title'].values:
        return f"Movie '{movie_title}' not found in the dataset."
    
    # Get the index of the movie
    movie_idx = movies[movies['title'] == movie_title].index[0]
    
    # Compute similarity only for this specific movie (much more memory efficient)
    movie_vector = tfidf_matrix[movie_idx]
    sim_scores = cosine_similarity(movie_vector, tfidf_matrix).flatten()
    
    # Get indices of movies sorted by similarity (excluding the movie itself)
    sim_indices = np.argsort(sim_scores)[::-1][1:top_n+1]
    
    # Return the top N similar movies with their similarity scores
    recommendations = []
    for idx in sim_indices:
        recommendations.append((movies.iloc[idx]['title'], sim_scores[idx]))
    
    return recommendations

# Test it out!
print("Recommendations for someone who liked 'Toy Story (1995)':")
recommendations = get_content_recommendations('Toy Story (1995)')
for movie, score in recommendations:
    print(f"{movie}: {score:.4f}")

Recommendations for someone who liked 'Toy Story (1995)':
The Snow Queen: Mirror Lands (2018): 1.0000
Puss in Book: Trapped in an Epic Tale (2017): 1.0000
Aladdin (1992): 1.0000
The Dragon Spell (2016): 1.0000
UglyDolls (2019): 1.0000


In [8]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Create a sample movie_matrix (user-item matrix)
# In a real scenario, this would be loaded from your dataset
np.random.seed(42)  # For reproducible results
users = [f'User_{i}' for i in range(1, 21)]  # 20 users
movies = ['Matrix, The (1999)', 'Titanic (1997)', 'Star Wars (1977)', 
          'Jurassic Park (1993)', 'Forrest Gump (1994)', 'Pulp Fiction (1994)',
          'The Shawshank Redemption (1994)', 'The Godfather (1972)']

# Create random ratings (1-5) with some NaN values to simulate missing ratings
movie_matrix = pd.DataFrame(np.random.choice([1, 2, 3, 4, 5, np.nan], 
                                           size=(len(users), len(movies)), 
                                           p=[0.1, 0.15, 0.2, 0.25, 0.2, 0.1]),
                          index=users, columns=movies)

# 1. Fill empty cells in our user-item matrix with 0 to perform math safely
# We transpose it (.T) because we want to calculate similarity BETWEEN MOVIES based on user ratings
movie_matrix_filled = movie_matrix.fillna(0).T

# 2. Compute similarity matrix based on user ratings
collaborative_sim = cosine_similarity(movie_matrix_filled)
collaborative_sim_df = pd.DataFrame(collaborative_sim, index=movie_matrix_filled.index, columns=movie_matrix_filled.index)

# Function to get collaborative recommendations
def get_collaborative_recommendations(movie_title, top_n=5):
    if movie_title not in collaborative_sim_df.columns:
        return f"Movie '{movie_title}' not found in the matrix."
    
    # Get user behavior similarity scores
    sim_scores = collaborative_sim_df[movie_title].sort_values(ascending=False)
    return sim_scores.iloc[1:top_n+1]

# Test it out!
print("\nRecommendations for someone who watched 'Matrix, The (1999)':")
print(get_collaborative_recommendations('Matrix, The (1999)'))


Recommendations for someone who watched 'Matrix, The (1999)':
Pulp Fiction (1994)                0.840209
Titanic (1997)                     0.828793
The Shawshank Redemption (1994)    0.823163
The Godfather (1972)               0.815976
Forrest Gump (1994)                0.694581
Name: Matrix, The (1999), dtype: float64


In [10]:
def recommend_for_user(user_id, top_n=5):
    # Check if user_id exists in the movie_matrix
    if user_id not in movie_matrix.index:
        available_users = movie_matrix.index.tolist()
        return f"User ID {user_id} not found. Available user IDs: {available_users[:10]}..."  # Show first 10 users
    
    # 1. Find movies this user has already rated highly (e.g., 4 stars or above)
    user_ratings = movie_matrix.loc[user_id].dropna()
    high_rated_movies = user_ratings[user_ratings >= 4.0].index.tolist()
    
    if not high_rated_movies:
        return "This user hasn't rated enough movies highly yet to build a profile."
    
    # 2. Accumulate similarity scores from all the movies they liked
    similar_scores = pd.Series(dtype='float64')
    for movie in high_rated_movies:
        if movie in collaborative_sim_df.columns:
            # Add up scores of movies similar to what they liked
            similar_scores = similar_scores.add(collaborative_sim_df[movie], fill_value=0)
            
    # 3. Sort the accumulated scores
    similar_scores = similar_scores.sort_values(ascending=False)
    
    # 4. Remove movies the user has already watched
    already_watched = user_ratings.index.tolist()
    recommendations = similar_scores.drop(labels=already_watched, errors='ignore')
    
    return recommendations.head(top_n)

# Generate custom recommendations for User ID #1
# First, let's check what user IDs are available
print("Available user IDs:", movie_matrix.index.tolist()[:10])  # Show first 10 users

print("\nTop 5 Personalized Recommendations for User 1:")
print(recommend_for_user(user_id=1))

Available user IDs: ['User_1', 'User_2', 'User_3', 'User_4', 'User_5', 'User_6', 'User_7', 'User_8', 'User_9', 'User_10']

Top 5 Personalized Recommendations for User 1:
User ID 1 not found. Available user IDs: ['User_1', 'User_2', 'User_3', 'User_4', 'User_5', 'User_6', 'User_7', 'User_8', 'User_9', 'User_10']...
